# Tema 6 - Introducción a los Sistemas de Control Realimentados

**Asignatura:** Fundamentos de Control - GIERM

---

## Objetivos de aprendizaje

1. Comprender la estructura de un sistema de control en lazo cerrado (realimentado).
2. Obtener las funciones de transferencia en lazo cerrado: referencia, perturbación y error.
3. Calcular el error en régimen permanente usando el Teorema del Valor Final.
4. Clasificar sistemas por **tipo** (número de integradores en lazo abierto).
5. Relacionar el tipo del sistema con el error ante entradas estándar (escalón, rampa, parábola).
6. Analizar el efecto de la acción integral sobre el rechazo de perturbaciones.
7. Comprender la función de sensibilidad y cómo la realimentación reduce la sensibilidad a cambios en la planta.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.lines import Line2D
import scipy.signal as signal

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

---

## 1. Diagrama de bloques del sistema realimentado

La estructura canónica de un sistema de control en lazo cerrado es:

- $R(s)$: referencia (entrada deseada)
- $E(s) = R(s) - B(s)$: señal de error
- $C(s)$: controlador
- $G(s)$: planta
- $Y(s)$: salida
- $H(s)$: sensor de realimentación
- $B(s) = H(s) Y(s)$: señal de realimentación
- $D(s)$: perturbación (entra entre controlador y planta)

In [ ]:
# Diagrama de bloques del sistema de control realimentado
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 6)
ax.set_aspect('equal')
ax.axis('off')

# Bloques
bloques = [
    (4, 3.5, 2, 1, r'$C(s)$'),
    (8, 3.5, 2, 1, r'$G(s)$'),
    (8, 1.0, 2, 1, r'$H(s)$'),
]
for (x, y, w, h, label) in bloques:
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                          facecolor='lightblue', edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=14, fontweight='bold')

# Nodo suma (comparador)
circle_sum = plt.Circle((3, 4), 0.35, fill=False, edgecolor='black', linewidth=1.5)
ax.add_patch(circle_sum)
ax.text(3, 4, r'$\Sigma$', ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(2.65, 4.25, '+', fontsize=10, color='green', fontweight='bold')
ax.text(2.65, 3.6, r'$-$', fontsize=10, color='red', fontweight='bold')

# Nodo suma perturbación
circle_pert = plt.Circle((7, 4), 0.35, fill=False, edgecolor='black', linewidth=1.5)
ax.add_patch(circle_pert)
ax.text(7, 4, r'$\Sigma$', ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(6.65, 4.25, '+', fontsize=10, color='green', fontweight='bold')
ax.text(7.25, 4.55, '+', fontsize=10, color='green', fontweight='bold')

# Flechas - cadena directa
ax.annotate('', xy=(2.65, 4), xytext=(1, 4),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
ax.annotate('', xy=(4, 4), xytext=(3.35, 4),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
ax.annotate('', xy=(6.65, 4), xytext=(6, 4),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
ax.annotate('', xy=(8, 4), xytext=(7.35, 4),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
ax.annotate('', xy=(13, 4), xytext=(10, 4),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))

# Perturbación D(s) entrando desde arriba
ax.annotate('', xy=(7, 4.35), xytext=(7, 5.5),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))

# Punto de bifurcación y realimentación
ax.plot(11.5, 4, 'ko', markersize=5)
ax.plot([11.5, 11.5], [4, 1.5], 'k-', lw=1.5)
ax.plot([11.5, 10], [1.5, 1.5], 'k-', lw=1.5)
ax.annotate('', xy=(8, 1.5), xytext=(10, 1.5),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
ax.plot([8, 3], [1.5, 1.5], 'k-', lw=1.5)
ax.annotate('', xy=(3, 3.65), xytext=(3, 1.5),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))

# Etiquetas de señales
ax.text(0.5, 4.3, r'$R(s)$', fontsize=13, fontweight='bold', color='darkblue')
ax.text(3.4, 4.4, r'$E(s)$', fontsize=12, color='darkred')
ax.text(6.2, 4.4, r'$U(s)$', fontsize=12, color='purple')
ax.text(7, 5.6, r'$D(s)$', fontsize=13, fontweight='bold', color='orange', ha='center')
ax.text(12.5, 4.3, r'$Y(s)$', fontsize=13, fontweight='bold', color='darkgreen')
ax.text(6, 1.1, r'$B(s) = H(s)Y(s)$', fontsize=11, color='gray')

ax.set_title('Diagrama de bloques: Sistema de control realimentado', fontsize=15, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

---

## 2. Funciones de transferencia en lazo cerrado

### 2.1 Derivación paso a paso

Partimos de las ecuaciones del diagrama de bloques:

1. Error: $E(s) = R(s) - H(s)\,Y(s)$
2. Salida del controlador: $U(s) = C(s)\,E(s)$
3. Salida de la planta: $Y(s) = G(s)\big[U(s) + D(s)\big] = G(s)\big[C(s)\,E(s) + D(s)\big]$

Sustituyendo (1) en (3):

$$Y(s) = G(s)\Big[C(s)\big(R(s) - H(s)Y(s)\big) + D(s)\Big]$$

$$Y(s) = C(s)G(s)R(s) - C(s)G(s)H(s)Y(s) + G(s)D(s)$$

$$Y(s)\big[1 + C(s)G(s)H(s)\big] = C(s)G(s)R(s) + G(s)D(s)$$

Definimos la **función de transferencia de lazo abierto** como:

$$G_{BA}(s) = C(s)\,G(s)\,H(s)$$

### 2.2 Función de transferencia de referencia

Con $D(s) = 0$:

$$Y(s) = \frac{C(s)G(s)}{1 + C(s)G(s)H(s)}\,R(s)$$

$$\boxed{G_{BC}(s) = \frac{C(s)\,G(s)}{1 + C(s)\,G(s)\,H(s)}}$$

### 2.3 Función de transferencia de perturbación

Con $R(s) = 0$:

$$Y(s) = \frac{G(s)}{1 + C(s)G(s)H(s)}\,D(s)$$

$$\boxed{G_{BC,\text{pert}}(s) = \frac{G(s)}{1 + C(s)\,G(s)\,H(s)}}$$

### 2.4 Función de transferencia del error

El error es $E(s) = R(s) - H(s)Y(s)$. Para $H(s) = 1$ (realimentación unitaria) y $D(s) = 0$:

$$E(s) = R(s) - Y(s) = R(s) - \frac{C(s)G(s)}{1 + C(s)G(s)} R(s)$$

$$E(s) = R(s)\left(1 - \frac{C(s)G(s)}{1 + C(s)G(s)}\right) = R(s)\cdot\frac{1}{1 + C(s)G(s)}$$

$$\boxed{E(s) = \frac{R(s)}{1 + C(s)\,G(s)\,H(s)}}$$

> **Nota:** En el caso general con $H(s) \neq 1$, el denominador es $1 + C(s)G(s)H(s)$ y el error se define como $E(s) = R(s) - B(s)$.

---

## 3. Error en régimen permanente: Teorema del Valor Final

Si el sistema es estable (todos los polos de $sE(s)$ en el semiplano izquierdo), el error en régimen permanente es:

$$\boxed{e_{rp} = \lim_{s \to 0} s\,E(s) = \lim_{s \to 0} \frac{s\,R(s)}{1 + G_{BA}(s)}}$$

donde $G_{BA}(s) = C(s)\,G(s)\,H(s)$ es la función de transferencia de lazo abierto.

### Cálculo para entradas estándar

| Entrada | $R(s)$ | $e_{rp}$ |
|---------|--------|----------|
| Escalón unitario | $\dfrac{1}{s}$ | $\dfrac{1}{1 + \lim_{s\to 0} G_{BA}(s)}$ |
| Rampa unitaria | $\dfrac{1}{s^2}$ | $\dfrac{1}{\lim_{s\to 0} s\,G_{BA}(s)}$ |
| Parábola unitaria | $\dfrac{1}{s^3}$ | $\dfrac{1}{\lim_{s\to 0} s^2\,G_{BA}(s)}$ |

---

## 4. Tipo de sistema y constantes de error estático

El **tipo** de un sistema se define como el **número de integradores** (polos en $s = 0$) en la función de transferencia de lazo abierto $G_{BA}(s) = C(s)G(s)H(s)$.

$$G_{BA}(s) = \frac{K \prod(s + z_i)}{s^N \prod(s + p_j)}$$

donde $N$ es el **tipo del sistema**.

### Constantes de error estático

- **Constante de posición:** $K_p = \lim_{s \to 0} G_{BA}(s)$
- **Constante de velocidad:** $K_v = \lim_{s \to 0} s\,G_{BA}(s)$
- **Constante de aceleración:** $K_a = \lim_{s \to 0} s^2\,G_{BA}(s)$

### 4.1 Sistema Tipo 0

$G_{BA}(s)$ no tiene polos en $s=0$. Ejemplo:

$$G_{BA}(s) = \frac{K}{s + a}$$

- $K_p = \lim_{s\to 0} \frac{K}{s+a} = \frac{K}{a}$ (finito)
- $K_v = \lim_{s\to 0} s \cdot \frac{K}{s+a} = 0$
- $K_a = 0$

Error ante escalón: $e_{rp} = \dfrac{1}{1 + K_p}$ (finito, **no cero**)

### 4.2 Sistema Tipo 1

$G_{BA}(s)$ tiene **un polo** en $s=0$. Ejemplo:

$$G_{BA}(s) = \frac{K}{s(s + a)}$$

- $K_p = \lim_{s\to 0} \frac{K}{s(s+a)} = \infty$
- $K_v = \lim_{s\to 0} s \cdot \frac{K}{s(s+a)} = \frac{K}{a}$ (finito)
- $K_a = 0$

Error ante escalón: $e_{rp} = \dfrac{1}{1 + \infty} = 0$ (**cero!**)

Error ante rampa: $e_{rp} = \dfrac{1}{K_v}$ (finito, **no cero**)

### 4.3 Sistema Tipo 2

$G_{BA}(s)$ tiene **dos polos** en $s=0$. Ejemplo:

$$G_{BA}(s) = \frac{K}{s^2(s + a)}$$

- $K_p = \infty$, $K_v = \infty$
- $K_a = \lim_{s\to 0} s^2 \cdot \frac{K}{s^2(s+a)} = \frac{K}{a}$ (finito)

Error ante escalón: $0$, error ante rampa: $0$, error ante parábola: $\dfrac{1}{K_a}$

### 4.4 Tabla resumen: Tipo de sistema vs error en régimen permanente

| Tipo del sistema | Entrada escalón $\frac{1}{s}$ | Entrada rampa $\frac{1}{s^2}$ | Entrada parábola $\frac{1}{s^3}$ |
|:---:|:---:|:---:|:---:|
| **Tipo 0** | $\dfrac{1}{1+K_p}$ | $\infty$ | $\infty$ |
| **Tipo 1** | $0$ | $\dfrac{1}{K_v}$ | $\infty$ |
| **Tipo 2** | $0$ | $0$ | $\dfrac{1}{K_a}$ |

> **Regla clave:** Un sistema de tipo $N$ tiene error **cero** ante entradas polinómicas de grado $< N$ y error **finito** ante la entrada de grado $N$. Para entradas de grado $> N$, el error es **infinito** (el sistema no sigue la referencia).

### 4.5 Gráfica: Respuesta al escalón - Tipo 0 vs Tipo 1

Comparamos la respuesta al escalón unitario de un sistema tipo 0 (con error permanente) y un sistema tipo 1 (error cero).

In [ ]:
# --- PLOT 1: Respuesta al escalón - Tipo 0 vs Tipo 1 ---
# Tipo 0: GBA = 4/(s+1) => GBC = 4/(s+5), Kp = 4, e_rp = 1/(1+4) = 0.2
# Tipo 1: GBA = 4/(s(s+1)) => GBC = 4/(s^2+s+4)

# Tipo 0: C(s)=4, G(s)=1/(s+1), H=1 => GBC = 4/((s+1)+4) = 4/(s+5)
num0 = [4]
den0 = [1, 5]
sys0 = signal.TransferFunction(num0, den0)

# Tipo 1: C(s)=4/s, G(s)=1/(s+1), H=1 => GBC = 4/(s(s+1)+4) = 4/(s^2+s+4)
num1 = [4]
den1 = [1, 1, 4]
sys1 = signal.TransferFunction(num1, den1)

t = np.linspace(0, 6, 1000)
_, y0 = signal.step(sys0, T=t)
_, y1 = signal.step(sys1, T=t)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tipo 0
axes[0].plot(t, np.ones_like(t), 'k--', linewidth=1.5, label=r'Referencia $r(t) = 1$')
axes[0].plot(t, y0, 'b-', linewidth=2, label=r'Salida $y(t)$ - Tipo 0')
axes[0].fill_between(t, y0, 1, alpha=0.15, color='red', label=r'Error $e_{rp} = 0.2$')
axes[0].axhline(y=0.8, color='red', linestyle=':', linewidth=1)
axes[0].annotate(r'$e_{rp} = \frac{1}{1+K_p} = 0.2$',
                 xy=(4.5, 0.85), fontsize=13, color='red', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='red'))
axes[0].set_title('Sistema Tipo 0 - Escalón', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel('Amplitud')
axes[0].legend(loc='right')
axes[0].set_ylim(-0.05, 1.3)
axes[0].grid(True, alpha=0.3)

# Tipo 1
axes[1].plot(t, np.ones_like(t), 'k--', linewidth=1.5, label=r'Referencia $r(t) = 1$')
axes[1].plot(t, y1, 'g-', linewidth=2, label=r'Salida $y(t)$ - Tipo 1')
axes[1].annotate(r'$e_{rp} = 0$ (error cero)',
                 xy=(4, 1.05), fontsize=13, color='green', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', edgecolor='green'))
axes[1].set_title('Sistema Tipo 1 - Escalón', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel('Amplitud')
axes[1].legend(loc='right')
axes[1].set_ylim(-0.05, 1.3)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Error en régimen permanente ante escalón: Tipo 0 vs Tipo 1',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.6 Gráfica: Respuesta a rampa - Tipo 1 vs Tipo 2

Comparamos la respuesta a una entrada rampa de un sistema tipo 1 (error constante) y tipo 2 (error cero).

In [ ]:
# --- PLOT 2: Respuesta a rampa - Tipo 1 vs Tipo 2 ---
# Tipo 1: GBA = 10/(s(s+2)), Kv = 10/2 = 5, e_rp_rampa = 1/Kv = 0.2
# Tipo 2: GBA = 10/(s^2(s+2)), Ka = 10/2 = 5, e_rp_rampa = 0

# Tipo 1: GBC = 10/(s^2 + 2s + 10)
num_t1 = [10]
den_t1 = [1, 2, 10]
sys_t1 = signal.TransferFunction(num_t1, den_t1)

# Tipo 2: GBC = 10/(s^3 + 2s^2 + 10)  
# GBA = 10/(s^2(s+2)), GBC = 10/(s^3 + 2s^2 + 10)
num_t2 = [10]
den_t2 = [1, 2, 0, 10]
sys_t2 = signal.TransferFunction(num_t2, den_t2)

# Verificar estabilidad tipo 2 - ajustar si es necesario
# Polos: s^3 + 2s^2 + 10 = 0 => inestable (falta término en s)
# Usamos GBA = 20(s+1)/(s^2(s+5)) para estabilidad
# GBC = 20(s+1) / (s^3 + 5s^2 + 20s + 20)
num_t2 = [20, 20]
den_t2 = [1, 5, 20, 20]
sys_t2 = signal.TransferFunction(num_t2, den_t2)

t = np.linspace(0, 8, 2000)
rampa = t

# Respuesta a rampa (convolución o lsim)
_, y_t1, _ = signal.lsim(sys_t1, rampa, t)
_, y_t2, _ = signal.lsim(sys_t2, rampa, t)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tipo 1
axes[0].plot(t, rampa, 'k--', linewidth=1.5, label=r'Referencia $r(t) = t$')
axes[0].plot(t, y_t1, 'b-', linewidth=2, label=r'Salida $y(t)$ - Tipo 1')
# Mostrar error constante
idx = -1
error_t1 = rampa - y_t1
axes[0].fill_between(t[500:], y_t1[500:], rampa[500:], alpha=0.15, color='red')
axes[0].annotate(r'$e_{rp} = \frac{1}{K_v} = 0.2$ (constante)',
                 xy=(5, 3.5), fontsize=12, color='red', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='red'))
axes[0].set_title('Sistema Tipo 1 - Rampa', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel('Amplitud')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Tipo 2
axes[1].plot(t, rampa, 'k--', linewidth=1.5, label=r'Referencia $r(t) = t$')
axes[1].plot(t, y_t2, 'g-', linewidth=2, label=r'Salida $y(t)$ - Tipo 2')
axes[1].annotate(r'$e_{rp} = 0$ (error cero)',
                 xy=(4.5, 2.5), fontsize=12, color='green', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', edgecolor='green'))
axes[1].set_title('Sistema Tipo 2 - Rampa', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel('Amplitud')
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Error en régimen permanente ante rampa: Tipo 1 vs Tipo 2',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.7 Ejemplo numérico: Cálculo de constantes de error

Sea $G_{BA}(s) = \dfrac{10(s+2)}{s(s+1)(s+5)}$ (sistema **tipo 1**).

- $K_p = \lim_{s\to 0} G_{BA}(s) = \lim_{s\to 0} \frac{10(s+2)}{s(s+1)(s+5)} = \infty$

- $K_v = \lim_{s\to 0} s \cdot G_{BA}(s) = \lim_{s\to 0} \frac{10(s+2)}{(s+1)(s+5)} = \frac{10 \cdot 2}{1 \cdot 5} = 4$

- $K_a = \lim_{s\to 0} s^2 \cdot G_{BA}(s) = \lim_{s\to 0} \frac{10s(s+2)}{(s+1)(s+5)} = 0$

**Errores:**
- Escalón: $e_{rp} = \dfrac{1}{1+K_p} = 0$
- Rampa: $e_{rp} = \dfrac{1}{K_v} = \dfrac{1}{4} = 0.25$
- Parábola: $e_{rp} = \dfrac{1}{K_a} = \infty$

$$\boxed{e_{rp,\text{escalón}} = 0, \quad e_{rp,\text{rampa}} = 0.25, \quad e_{rp,\text{parábola}} = \infty}$$

In [ ]:
# --- PLOT 3: Verificación numérica del ejemplo ---
# GBA = 10(s+2) / (s(s+1)(s+5))
# GBC = 10(s+2) / (s^3 + 6s^2 + 15s + 20)
# Denominador: s(s+1)(s+5) + 10(s+2) = s^3 + 6s^2 + 5s + 10s + 20 = s^3 + 6s^2 + 15s + 20

num_ej = [10, 20]
den_ej = [1, 6, 15, 20]
sys_ej = signal.TransferFunction(num_ej, den_ej)

t = np.linspace(0, 10, 2000)
rampa = t

_, y_step = signal.step(sys_ej, T=t)
_, y_ramp, _ = signal.lsim(sys_ej, rampa, t)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Escalón
axes[0].plot(t, np.ones_like(t), 'k--', linewidth=1.5, label='Referencia')
axes[0].plot(t, y_step, 'b-', linewidth=2, label=r'Salida $y(t)$')
axes[0].set_title(r'Tipo 1: Escalón $\Rightarrow e_{rp} = 0$', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel('Amplitud')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-0.05, 1.3)

# Rampa
axes[1].plot(t, rampa, 'k--', linewidth=1.5, label=r'Referencia $r(t)=t$')
axes[1].plot(t, y_ramp, 'b-', linewidth=2, label=r'Salida $y(t)$')
error_ramp = rampa - y_ramp
axes[1].fill_between(t[300:], y_ramp[300:], rampa[300:], alpha=0.15, color='red')
axes[1].annotate(r'$e_{rp} = 1/K_v = 0.25$',
                 xy=(6, 4), fontsize=13, color='red', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='red'))
axes[1].set_title(r'Tipo 1: Rampa $\Rightarrow e_{rp} = 1/K_v = 0.25$', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel('Amplitud')
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.suptitle(r'Verificación numérica: $G_{BA}(s) = \frac{10(s+2)}{s(s+1)(s+5)}$ (Tipo 1)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 5. Error ante perturbaciones y acción integral

La salida ante una perturbación escalón $D(s) = 1/s$ con $R(s) = 0$ es:

$$Y(s) = \frac{G(s)}{1 + C(s)G(s)H(s)} \cdot \frac{1}{s}$$

El error en régimen permanente debido a la perturbación (con $H(s)=1$):

$$e_{rp,\text{pert}} = -\lim_{s \to 0} s \cdot Y(s) = -\frac{G(0)}{1 + C(0)G(0)}$$

Si $C(s)$ contiene **acción integral** ($C(s) = \frac{\ldots}{s}$), entonces $C(0) \to \infty$ y:

$$\boxed{e_{rp,\text{pert}} = -\frac{G(0)}{1 + \infty} = 0}$$

**Conclusión:** La acción integral en el controlador **elimina el error en régimen permanente** ante perturbaciones escalón.

In [ ]:
# --- PLOT 4: Efecto de la acción integral ante perturbación ---
# Planta: G(s) = 1/(s+1)
# Sin integral: C(s) = 5 (proporcional puro)
# Con integral: C(s) = 5 + 3/s = (5s+3)/s (PI)

# Referencia escalón + perturbación escalón en t=5
t = np.linspace(0, 15, 3000)
dt = t[1] - t[0]

# Señal de referencia (escalón en t=0) y perturbación (escalón en t=5)
r = np.ones_like(t)
d = np.zeros_like(t)
d[t >= 5] = 0.5  # perturbación de amplitud 0.5

# --- Sin acción integral: C=5, G=1/(s+1) ---
# GBC_ref = 5/(s+6), GBC_pert = 1/(s+6)
sys_ref_P = signal.TransferFunction([5], [1, 6])
sys_pert_P = signal.TransferFunction([1], [1, 6])

_, y_ref_P, _ = signal.lsim(sys_ref_P, r, t)
_, y_pert_P, _ = signal.lsim(sys_pert_P, d, t)
y_total_P = y_ref_P + y_pert_P

# --- Con acción integral: C=(5s+3)/s, G=1/(s+1) ---
# GBA = (5s+3)/(s(s+1))
# GBC_ref = (5s+3)/(s^2+s+5s+3) = (5s+3)/(s^2+6s+3)
# GBC_pert = s/((s+1)(s) + (5s+3)) = s/(s^2+6s+3)
sys_ref_PI = signal.TransferFunction([5, 3], [1, 6, 3])
sys_pert_PI = signal.TransferFunction([1, 0], [1, 6, 3])

_, y_ref_PI, _ = signal.lsim(sys_ref_PI, r, t)
_, y_pert_PI, _ = signal.lsim(sys_pert_PI, d, t)
y_total_PI = y_ref_PI + y_pert_PI

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sin integral
axes[0].plot(t, r, 'k--', linewidth=1.5, label='Referencia')
axes[0].plot(t, y_total_P, 'r-', linewidth=2, label=r'Salida con $C=5$ (solo P)')
axes[0].axvline(x=5, color='orange', linestyle=':', linewidth=1.5, label='Perturbación en t=5')
axes[0].annotate(r'Error por perturbación $\neq 0$',
                 xy=(10, 0.87), fontsize=11, color='red', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='red'))
axes[0].annotate(r'Error por escalón $\neq 0$',
                 xy=(2, 0.7), fontsize=11, color='darkred',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='darkred'))
axes[0].set_title('Sin acción integral (C = Kp)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel('Amplitud')
axes[0].legend(loc='lower right', fontsize=10)
axes[0].set_ylim(-0.05, 1.3)
axes[0].grid(True, alpha=0.3)

# Con integral
axes[1].plot(t, r, 'k--', linewidth=1.5, label='Referencia')
axes[1].plot(t, y_total_PI, 'g-', linewidth=2, label=r'Salida con $C=5+3/s$ (PI)')
axes[1].axvline(x=5, color='orange', linestyle=':', linewidth=1.5, label='Perturbación en t=5')
axes[1].annotate(r'Error $\to 0$ (integral elimina error)',
                 xy=(9, 1.1), fontsize=11, color='green', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', edgecolor='green'))
axes[1].set_title('Con acción integral (C = PI)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel('Amplitud')
axes[1].legend(loc='lower right', fontsize=10)
axes[1].set_ylim(-0.05, 1.3)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Efecto de la acción integral en el rechazo de perturbaciones',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 6. Sensibilidad

La **función de sensibilidad** mide cómo afectan los cambios en la planta $G(s)$ a la salida del sistema en lazo cerrado.

Para la función de transferencia de lazo cerrado $G_{BC} = \dfrac{C\,G}{1 + C\,G\,H}$, la sensibilidad respecto a $G$ es:

$$S_G^{G_{BC}} = \frac{\partial G_{BC} / G_{BC}}{\partial G / G} = \frac{G}{G_{BC}} \cdot \frac{\partial G_{BC}}{\partial G}$$

Calculando:

$$\frac{\partial G_{BC}}{\partial G} = \frac{C(1+CGH) - CG \cdot CH}{(1+CGH)^2} = \frac{C}{(1+CGH)^2}$$

$$S_G^{G_{BC}} = \frac{G}{\frac{CG}{1+CGH}} \cdot \frac{C}{(1+CGH)^2} = \frac{1+CGH}{C} \cdot \frac{C}{(1+CGH)^2}$$

$$\boxed{S(s) = \frac{1}{1 + G_{BA}(s)}}$$

Para $|G_{BA}(j\omega)| \gg 1$ (ganancia de lazo alta), $S \approx 0$: la realimentación **reduce la sensibilidad** a cambios en la planta.

In [ ]:
# --- PLOT 5: Efecto de la sensibilidad - variación de la planta ---
# Planta nominal: G(s) = 2/(s+2)
# Planta modificada: G'(s) = 3/(s+2) (variación del 50%)
# Controlador: C(s) = 10

# Lazo abierto: Y = G*R (sin realimentación)
G_nom = signal.TransferFunction([2], [1, 2])
G_mod = signal.TransferFunction([3], [1, 2])

# Lazo cerrado: Y = CG/(1+CG) * R
# Nominal: 20/(s+22)
GBC_nom = signal.TransferFunction([20], [1, 22])
# Modificada: 30/(s+32)
GBC_mod = signal.TransferFunction([30], [1, 32])

t = np.linspace(0, 4, 1000)
_, y_oa_nom = signal.step(G_nom, T=t)
_, y_oa_mod = signal.step(G_mod, T=t)
_, y_lc_nom = signal.step(GBC_nom, T=t)
_, y_lc_mod = signal.step(GBC_mod, T=t)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lazo abierto
axes[0].plot(t, y_oa_nom, 'b-', linewidth=2, label=r'$G = 2/(s+2)$ (nominal)')
axes[0].plot(t, y_oa_mod, 'r--', linewidth=2, label=r'$G = 3/(s+2)$ (+50%)')
axes[0].fill_between(t, y_oa_nom, y_oa_mod, alpha=0.2, color='red')
diff_oa = abs(y_oa_nom[-1] - y_oa_mod[-1])
axes[0].annotate(f'Diferencia en r.p. = {diff_oa:.2f}\n(50% de variación)',
                 xy=(2.5, 0.75), fontsize=11, color='red', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='red'))
axes[0].set_title('Lazo abierto: Alta sensibilidad', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel('Amplitud')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-0.05, 1.7)

# Lazo cerrado
axes[1].plot(t, y_lc_nom, 'b-', linewidth=2, label=r'$G$ nominal')
axes[1].plot(t, y_lc_mod, 'r--', linewidth=2, label=r'$G$ +50%')
axes[1].fill_between(t, y_lc_nom, y_lc_mod, alpha=0.2, color='green')
diff_lc = abs(y_lc_nom[-1] - y_lc_mod[-1])
axes[1].annotate(f'Diferencia en r.p. = {diff_lc:.3f}\n(variación reducida!)',
                 xy=(2, 0.75), fontsize=11, color='green', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', edgecolor='green'))
axes[1].set_title('Lazo cerrado: Baja sensibilidad', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel('Amplitud')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-0.05, 1.7)

plt.suptitle(r'Sensibilidad: Efecto de variación del 50% en $G(s)$',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- PLOT 6: Función de sensibilidad en frecuencia ---
# S(jw) = 1 / (1 + GBA(jw))
# GBA = 10*2/(s+2) = 20/(s+2)

w = np.logspace(-2, 3, 1000)
s = 1j * w

GBA_jw = 20 / (s + 2)
S_jw = 1 / (1 + GBA_jw)

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(w, 20*np.log10(np.abs(S_jw)), 'b-', linewidth=2, label=r'$|S(j\omega)|$ (dB)')
ax.axhline(y=0, color='k', linestyle='--', linewidth=1)
ax.axhline(y=-20, color='gray', linestyle=':', linewidth=1)
ax.fill_between(w, -60, 20*np.log10(np.abs(S_jw)), alpha=0.1, color='blue')
ax.annotate(r'Zona de baja sensibilidad $|S| \ll 1$',
            xy=(0.1, -18), fontsize=12, color='blue', fontweight='bold')
ax.annotate(r'$|S| \to 1$ a altas frecuencias',
            xy=(50, -2), fontsize=11, color='red')
ax.set_xlabel(r'Frecuencia $\omega$ (rad/s)')
ax.set_ylabel('Magnitud (dB)')
ax.set_title(r'Función de sensibilidad $S(j\omega) = \frac{1}{1+G_{BA}(j\omega)}$',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, which='both')
ax.set_ylim(-35, 5)
plt.tight_layout()
plt.show()

---

## 7. Ejercicios resueltos

#### Ejercicio resuelto 1: Función de transferencia en lazo cerrado

**Enunciado:** Dado un sistema con $C(s) = 5$, $G(s) = \dfrac{1}{s+3}$ y $H(s) = 1$, obtener $G_{BC}(s)$.

**Solución:**

$$G_{BC}(s) = \frac{C(s)G(s)}{1 + C(s)G(s)H(s)} = \frac{\frac{5}{s+3}}{1 + \frac{5}{s+3}} = \frac{5}{s+3+5} = \frac{5}{s+8}$$

$$\boxed{G_{BC}(s) = \frac{5}{s+8}}$$

Polo en lazo cerrado: $s = -8$ (estable, más rápido que el polo original $s=-3$).

#### Ejercicio resuelto 2: Tipo del sistema y errores

**Enunciado:** Sea $G_{BA}(s) = \dfrac{20}{s(s+4)}$. Determinar el tipo del sistema y el error ante escalón y rampa.

**Solución:**

El sistema tiene **un polo en $s=0$** $\Rightarrow$ **Tipo 1**.

- $K_p = \lim_{s\to 0} \frac{20}{s(s+4)} = \infty$ $\Rightarrow$ $e_{rp,\text{escalón}} = \frac{1}{1+\infty} = 0$

- $K_v = \lim_{s\to 0} s \cdot \frac{20}{s(s+4)} = \frac{20}{4} = 5$ $\Rightarrow$ $e_{rp,\text{rampa}} = \frac{1}{K_v} = \frac{1}{5} = 0.2$

$$\boxed{\text{Tipo 1:}\quad e_{rp,\text{escalón}} = 0, \quad e_{rp,\text{rampa}} = 0.2}$$

#### Ejercicio resuelto 3: Encontrar $K$ para un error deseado

**Enunciado:** Sea $G_{BA}(s) = \dfrac{K}{s(s+2)(s+10)}$. Determinar $K$ para que el error ante rampa sea $e_{rp} = 0.05$.

**Solución:**

Sistema **Tipo 1** (un integrador).

$$K_v = \lim_{s\to 0} s \cdot \frac{K}{s(s+2)(s+10)} = \frac{K}{2 \cdot 10} = \frac{K}{20}$$

$$e_{rp} = \frac{1}{K_v} = \frac{20}{K} = 0.05$$

$$\boxed{K = \frac{20}{0.05} = 400}$$

#### Ejercicio resuelto 4: Error ante perturbación

**Enunciado:** Planta $G(s) = \dfrac{1}{s+1}$, controlador $C(s) = K_p$, realimentación unitaria. Una perturbación escalón $D(s) = \dfrac{1}{s}$ actúa a la entrada de la planta. Calcular el error en r.p. debido a la perturbación. ¿Qué ocurre si $C(s) = K_p + \dfrac{K_i}{s}$?

**Solución:**

Con $R(s) = 0$, la salida debida a la perturbación es:

$$Y_{pert}(s) = \frac{G(s)}{1 + C(s)G(s)} D(s)$$

**Caso 1: $C(s) = K_p$**

$$y_{pert,rp} = \lim_{s\to 0} s \cdot \frac{\frac{1}{s+1}}{1 + \frac{K_p}{s+1}} \cdot \frac{1}{s} = \frac{1}{1+K_p}$$

Para $K_p = 5$: $y_{pert,rp} = \frac{1}{6} \approx 0.167$ (error permanente $\neq 0$).

**Caso 2: $C(s) = K_p + K_i/s$**

$$y_{pert,rp} = \lim_{s\to 0} \frac{s \cdot \frac{1}{s+1}}{1 + \frac{K_p s + K_i}{s(s+1)}} \cdot \frac{1}{s} = \lim_{s\to 0} \frac{s}{s(s+1) + K_p s + K_i} = \frac{0}{K_i} = 0$$

$$\boxed{\text{Con PI: } y_{pert,rp} = 0 \quad \text{(la acción integral elimina el error por perturbación)}}$$

#### Ejercicio resuelto 5: Efecto de $H(s) \neq 1$

**Enunciado:** Sea $C(s) = 10$, $G(s) = \dfrac{1}{s+2}$, $H(s) = 0.5$. Obtener $G_{BC}(s)$ y calcular el valor final ante escalón unitario.

**Solución:**

$$G_{BC}(s) = \frac{C(s)G(s)}{1 + C(s)G(s)H(s)} = \frac{\frac{10}{s+2}}{1 + \frac{10 \cdot 0.5}{s+2}} = \frac{10}{s+2+5} = \frac{10}{s+7}$$

Valor final ante escalón:

$$y_{rp} = \lim_{s\to 0} s \cdot \frac{10}{s+7} \cdot \frac{1}{s} = \frac{10}{7} \approx 1.429$$

**Atención:** Con $H(s) \neq 1$, la salida **no converge a la referencia** aunque el sistema sea tipo 1 en lazo abierto.

$$\boxed{G_{BC}(s) = \frac{10}{s+7}, \quad y_{rp} = \frac{10}{7} \approx 1.43}$$

#### Ejercicio resuelto 6: Estabilidad desde polos en lazo cerrado

**Enunciado:** Sea $G_{BA}(s) = \dfrac{K}{s(s+3)(s+5)}$. Determinar el rango de $K$ para estabilidad usando la ecuación característica.

**Solución:**

Ecuación característica: $1 + G_{BA}(s) = 0$

$$s(s+3)(s+5) + K = 0 \Rightarrow s^3 + 8s^2 + 15s + K = 0$$

Criterio de Routh:

| $s^3$ | 1 | 15 |
|:---:|:---:|:---:|
| $s^2$ | 8 | K |
| $s^1$ | $\frac{120 - K}{8}$ | 0 |
| $s^0$ | K | |

Condiciones: $\frac{120-K}{8} > 0 \Rightarrow K < 120$ y $K > 0$.

$$\boxed{0 < K < 120}$$

In [ ]:
# --- PLOT 7: Verificación de estabilidad para distintos K ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
t = np.linspace(0, 10, 2000)

K_values = [50, 120, 150]
colores = ['green', 'orange', 'red']
titulos = [r'$K=50$ (Estable)', r'$K=120$ (Límite)', r'$K=150$ (Inestable)']

for i, (K, color, titulo) in enumerate(zip(K_values, colores, titulos)):
    # GBC = K / (s^3 + 8s^2 + 15s + K)
    sys_k = signal.TransferFunction([K], [1, 8, 15, K])
    try:
        _, y_k = signal.step(sys_k, T=t)
        axes[i].plot(t, y_k, color=color, linewidth=2)
        axes[i].axhline(y=1, color='k', linestyle='--', linewidth=1)
    except:
        axes[i].text(5, 0.5, 'Inestable', ha='center', fontsize=14, color='red')
    axes[i].set_title(titulo, fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Tiempo (s)')
    axes[i].set_ylabel('Amplitud')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_ylim(-3, 5)

plt.suptitle(r'Respuesta al escalón para $G_{BA} = K/[s(s+3)(s+5)]$: Efecto de $K$',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

---

## 8. Catálogo completo de ejercicios: todos los patrones

### 8.1 Tipo 1: Obtener la función de transferencia en lazo cerrado

**Enunciado:** Dado $C(s) = 8$, $G(s) = \dfrac{1}{(s+1)(s+4)}$, $H(s) = 1$, obtener $G_{BC}(s)$.

**Solución:**

$$G_{BC}(s) = \frac{8}{(s+1)(s+4) + 8} = \frac{8}{s^2 + 5s + 4 + 8} = \frac{8}{s^2 + 5s + 12}$$

$$\boxed{G_{BC}(s) = \frac{8}{s^2 + 5s + 12}}$$

### 8.2 Tipo 2: Error ante escalón para sistema tipo 0

**Enunciado:** Sea $G_{BA}(s) = \dfrac{9}{(s+1)(s+3)}$. Calcular $e_{rp}$ ante escalón.

**Solución:**

Tipo 0 (sin integradores). $K_p = \lim_{s\to 0} \frac{9}{(s+1)(s+3)} = \frac{9}{3} = 3$

$$\boxed{e_{rp} = \frac{1}{1+K_p} = \frac{1}{1+3} = 0.25}$$

### 8.3 Tipo 3: Error ante rampa para sistema tipo 1

**Enunciado:** Sea $G_{BA}(s) = \dfrac{50}{s(s+5)(s+10)}$. Calcular $e_{rp}$ ante rampa.

**Solución:**

Tipo 1. $K_v = \lim_{s\to 0} s \cdot \frac{50}{s(s+5)(s+10)} = \frac{50}{50} = 1$

$$\boxed{e_{rp,\text{rampa}} = \frac{1}{K_v} = 1}$$

### 8.4 Tipo 4: Error ante parábola para sistema tipo 2

**Enunciado:** Sea $G_{BA}(s) = \dfrac{100(s+1)}{s^2(s+5)(s+10)}$. Calcular $e_{rp}$ ante parábola.

**Solución:**

Tipo 2 (dos integradores). 

$$K_a = \lim_{s\to 0} s^2 \cdot \frac{100(s+1)}{s^2(s+5)(s+10)} = \frac{100 \cdot 1}{5 \cdot 10} = 2$$

$$\boxed{e_{rp,\text{parábola}} = \frac{1}{K_a} = \frac{1}{2} = 0.5}$$

### 8.5 Tipo 5: Determinar el tipo del sistema

**Enunciado:** Clasificar los siguientes sistemas:

a) $G_{BA}(s) = \dfrac{5(s+2)}{(s+1)(s+10)}$ $\Rightarrow$ **Tipo 0** (sin polos en $s=0$)

b) $G_{BA}(s) = \dfrac{8}{s(s^2+4s+8)}$ $\Rightarrow$ **Tipo 1** (un polo en $s=0$)

c) $G_{BA}(s) = \dfrac{K(s+3)}{s^2(s+7)}$ $\Rightarrow$ **Tipo 2** (dos polos en $s=0$)

### 8.6 Tipo 6: Encontrar $K$ para un error especificado

**Enunciado:** Sea $G_{BA}(s) = \dfrac{K(s+1)}{s(s+2)(s+8)}$. Encontrar $K$ para que $e_{rp,\text{rampa}} \leq 0.1$.

**Solución:**

Tipo 1. $K_v = \lim_{s\to 0} \frac{K(s+1)}{(s+2)(s+8)} = \frac{K}{16}$

$$e_{rp} = \frac{1}{K_v} = \frac{16}{K} \leq 0.1 \Rightarrow K \geq 160$$

$$\boxed{K \geq 160}$$

### 8.7 Tipo 7: Rechazo de perturbaciones

**Enunciado:** $G(s) = \dfrac{1}{s+2}$, $C(s) = 10$, perturbación escalón $D = 0.5$. Calcular efecto en r.p.

**Solución:**

$$y_{pert,rp} = \frac{G(0)}{1+C(0)G(0)} \cdot D = \frac{0.5}{1 + 10 \cdot 0.5} \cdot 0.5 = \frac{0.5}{6} \cdot 0.5 = \frac{0.25}{6} \approx 0.042$$

$$\boxed{y_{pert,rp} \approx 0.042}$$

### 8.8 Tipo 8: Sensibilidad

**Enunciado:** Si $|G_{BA}(j\omega_0)| = 100$ a cierta frecuencia, ¿cuánto varía la salida si la planta cambia un 10%?

**Solución:**

$$|S(j\omega_0)| = \frac{1}{|1 + G_{BA}(j\omega_0)|} \approx \frac{1}{|G_{BA}(j\omega_0)|} = \frac{1}{100} = 0.01$$

Variación en la salida $\approx 0.01 \times 10\% = 0.1\%$.

$$\boxed{\Delta y \approx 0.1\%}$$

### 8.9 Tipo 9: Efecto de $H(s) \neq 1$

**Enunciado:** $C(s) = \dfrac{5}{s}$, $G(s) = \dfrac{1}{s+3}$, $H(s) = 2$. Calcular el valor final ante escalón.

**Solución:**

$$G_{BC}(s) = \frac{\frac{5}{s(s+3)}}{1 + \frac{10}{s(s+3)}} = \frac{5}{s^2 + 3s + 10}$$

$$y_{rp} = \lim_{s\to 0} s \cdot \frac{5}{s^2+3s+10} \cdot \frac{1}{s} = \frac{5}{10} = 0.5$$

**Nota:** La referencia es 1, pero la salida converge a 0.5 porque $H(s) = 2 \neq 1$. El valor final es $\frac{1}{H(0)} = 0.5$ cuando $G_{BA}$ tiene integrador.

$$\boxed{y_{rp} = 0.5 \neq 1 \quad (\text{requiere precompensación o } H=1)}$$

### 8.10 Tipo 10: Estabilidad desde los polos en lazo cerrado

**Enunciado:** Sea $G_{BC}(s) = \dfrac{10}{s^2 + 2s + 10}$. ¿Es estable el sistema? ¿Cuál es el error ante escalón?

**Solución:**

Polos: $s = \frac{-2 \pm \sqrt{4-40}}{2} = -1 \pm 3j$

Parte real negativa $\Rightarrow$ **sistema estable**.

Valor final ante escalón: $y_{rp} = G_{BC}(0) = \frac{10}{10} = 1$ $\Rightarrow$ $e_{rp} = 0$.

$$\boxed{\text{Estable (polos en } s = -1 \pm 3j\text{), } e_{rp} = 0}$$

---

## 9. Resumen y fórmulas clave

### Funciones de transferencia en lazo cerrado

| Función | Expresión |
|:--------|:----------|
| Lazo abierto | $G_{BA}(s) = C(s)\,G(s)\,H(s)$ |
| Referencia | $G_{BC}(s) = \dfrac{C(s)G(s)}{1 + G_{BA}(s)}$ |
| Perturbación | $G_{BC,pert}(s) = \dfrac{G(s)}{1 + G_{BA}(s)}$ |
| Error ($H=1$) | $E(s) = \dfrac{R(s)}{1 + C(s)G(s)}$ |

### Error en régimen permanente

$$e_{rp} = \lim_{s \to 0} \frac{s\,R(s)}{1 + G_{BA}(s)}$$

### Constantes de error estático

| Constante | Definición | Sistema tipo 0 | Tipo 1 | Tipo 2 |
|:---------:|:----------:|:----:|:----:|:----:|
| $K_p$ | $\lim_{s\to 0} G_{BA}(s)$ | Finita | $\infty$ | $\infty$ |
| $K_v$ | $\lim_{s\to 0} s\,G_{BA}(s)$ | 0 | Finita | $\infty$ |
| $K_a$ | $\lim_{s\to 0} s^2 G_{BA}(s)$ | 0 | 0 | Finita |

### Tabla maestra: Tipo vs Error

| Tipo | Escalón | Rampa | Parábola |
|:----:|:-------:|:-----:|:--------:|
| 0 | $\dfrac{1}{1+K_p}$ | $\infty$ | $\infty$ |
| 1 | $0$ | $\dfrac{1}{K_v}$ | $\infty$ |
| 2 | $0$ | $0$ | $\dfrac{1}{K_a}$ |

### Sensibilidad

$$S(s) = \frac{1}{1 + G_{BA}(s)}$$

### Reglas clave

- La **acción integral** ($1/s$ en $C(s)$) aumenta el tipo del sistema y elimina el error ante perturbaciones escalón.
- Con $H(s) \neq 1$, el valor final no es igual a la referencia aunque el sistema sea tipo $\geq 1$.
- **Más ganancia** $K$ $\Rightarrow$ menor error, pero puede comprometer la estabilidad.
- La realimentación reduce la sensibilidad a variaciones de la planta en la banda de frecuencias donde $|G_{BA}| \gg 1$.